In [1]:
!pip install -r ../requirements.txt

In [2]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from dotenv import load_dotenv


c:\Users\livex\Desktop\Uni\FitRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CONFIG_PATH = "../data/processed/retriever_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

print("✅ Retriever config loaded:")
print(json.dumps(config, indent=2))


✅ Retriever config loaded:
{
  "model_name": "multi-qa-MiniLM-L6-cos-v1",
  "index_path": "../embeddings/vector_store",
  "search_type": "mmr",
  "k": 5,
  "fetch_k": 20,
  "lambda_mult": 0.5,
  "total_chunks": 2997,
  "total_vectors": 2997,
  "embedding_dims": 384
}


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name=config["model_name"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.load_local(
    config["index_path"],
    embedding_model,
    allow_dangerous_deserialization=True
)

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k"           : config["k"],
        "fetch_k"     : config["fetch_k"],
        "lambda_mult" : config["lambda_mult"]
    }
)

print(f"✅ Embedding model loaded  : {config['model_name']}")
print(f"✅ FAISS index loaded      : {config['total_vectors']} vectors, {config['embedding_dims']} dims")
print(f"✅ MMR retriever ready     : k={config['k']}, fetch_k={config['fetch_k']}, lambda={config['lambda_mult']}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9648.75it/s]


✅ Embedding model loaded  : multi-qa-MiniLM-L6-cos-v1
✅ FAISS index loaded      : 2997 vectors, 384 dims
✅ MMR retriever ready     : k=5, fetch_k=20, lambda=0.5


In [6]:
load_dotenv("../.env")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

# Install if needed: pip install langchain-groq
test_response = llm.invoke("Reply with exactly: LLM connected.")
print(f"✅ LLM connected: {test_response.content}")
print(f"   Model : llama-3.3-70b-versatile via Groq")

✅ LLM connected: LLM connected.
   Model : llama-3.3-70b-versatile via Groq


In [ ]:
SYSTEM_PROMPT = """You are FitRAG, an expert fitness assistant for beginners.
You answer questions about exercise, training programs, and physical fitness.

STRICT RULES:
1. Answer ONLY using the information provided in the context below.
2. Do NOT use any knowledge from your training data — only the context.
3. If the context does not contain enough information to answer, say exactly: 
"I don't have enough information in my knowledge base to answer this question."
4. When possible, mention which source your answer comes from.
5. Keep answers clear and beginner-friendly.
6. Do not make up specific numbers, studies, or recommendations not in the context.
7. Dont mention the source except at the end

Context:
{context}
"""

HUMAN_PROMPT = "Question: {question}"

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  HUMAN_PROMPT)
])

print("✅ Prompt template built")
print("\n--- System prompt preview ---")
print(SYSTEM_PROMPT[:400] + "...")


✅ Prompt template built

--- System prompt preview ---
You are FitRAG, an expert fitness assistant for beginners.
You answer questions about exercise, training programs, and physical fitness.

STRICT RULES:
1. Answer ONLY using the information provided in the context below.
2. Do NOT use any knowledge from your training data — only the context.
3. If the context does not contain enough information to answer, say exactly: "I don't have enough informati...


In [22]:
def format_context(docs):
    formatted = []
    for i, doc in enumerate(docs):
        source = os.path.basename(doc.metadata.get("source", "Unknown"))
        page   = doc.metadata.get("page", "?")
        formatted.append(
            f"[Source {i+1}: {source}, page {page}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

# Build the RAG chain
rag_chain = (
    {
        "context" : mmr_retriever | format_context,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built successfully")
print("   Pipeline: MMR retriever → prompt template → Groq → string output")


✅ RAG chain built successfully
   Pipeline: MMR retriever → prompt template → Groq → string output


In [49]:
def run_rag(query, show_context=True):
    """
    Run the full RAG pipeline and display:
    - The retrieved chunks (context given to the LLM)
    - The generated answer
    """

    # Step 1 — retrieve
    retrieved_docs = mmr_retriever.invoke(query)

    if show_context:
        print(f"\n📂 RETRIEVED CONTEXT ({len(retrieved_docs)} chunks):")
        print("-" * 65)
        for i, doc in enumerate(retrieved_docs):
            source = os.path.basename(doc.metadata.get("source", "Unknown"))
            page   = doc.metadata.get("page", "?")
            score_docs = vectorstore.similarity_search_with_score(query, k=1)
            print(f"Chunk #{i+1} | {source} — page {page}")
            print(f"  {doc.page_content[:220]}...")
            print("-" * 65)

    # Step 2 — generate
    answer = rag_chain.invoke(query)

    print(f"\n🤖 GENERATED ANSWER:")
    print("-" * 65)
    print(answer)
    print()
    return answer


In [50]:
# ── Query 1: Clear in-scope ──────────────────────────────────────────────
answer1 = run_rag("How many days per week should a beginner train?")



📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------
Chunk #1 | SSW.pdf — page 2
  Who Wants to Be a Novice?
3 StartingStrength.com© 2013 The Aasgaard Company
so he’s not really capable of inflicting enough training stress in a sane workout to prevent his recovery 
in a short period of time. This time ...
-----------------------------------------------------------------
Chunk #2 | NSCA_5.pdf — page 11
  movements that should be performed at a high velocity.
Although additional research is needed, it is likely that the
performance of different training velocities within a training
program may provide the most effective r...
-----------------------------------------------------------------
Chunk #3 | Increasing_Anaerobic_Endurance_Using_Strength_Endu.pdf — page 6
  training cycle that uses a weekly timeframe, also known as weekly training (Lorenz & Morrison, 
2015). This study used continuous running with power endurance and continuous runnin

In [52]:
# ── Query 2: Clear in-scope ──────────────────────────────────────────────
answer2 = run_rag("What is progressive overload and why does it matter for beginners?")



📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------
Chunk #1 | ProgressiveOverloadinLong-TermExerciseInterventionsTargetingExecutiveFunction.pdf — page 5
  For Peer Review
PROGRESSIVE OVERLOAD AND EXECUTIVE FUNCTION 5
1 forms of structured physical activity and sport that are designed to promote adaptation and enhance EF. 
2 Progressive overload is typically defined by the ...
-----------------------------------------------------------------
Chunk #2 | ProgressiveOverloadinLong-TermExerciseInterventionsTargetingExecutiveFunction.pdf — page 11
  10 memory and inhibitory control. Third, inconsistent reporting of progressive overload parameters, the 
11 absence of studies that examined progressive overload as an independent variable, and limited research 
12 apply...
-----------------------------------------------------------------
Chunk #3 | progressive_overload.pdf — page 4
  from different forms of stress. In some cases the speed
presents 

In [53]:
# ── Query 3: Multi-concept (requires synthesising chunks) ────────────────
answer3 = run_rag("How should a beginner structure a full week of training including rest days?")



📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------
Chunk #1 | WHO.pdf — page 19
  activity, across the week.
Strong recommendation, moderate certainty evidence
 Vigorous-intensity aerobic 
activities, as well as those that 
strengthen muscle and bone 
should be incorporated at least 
3 days a week.
St...
-----------------------------------------------------------------
Chunk #2 | NSCA_4.pdf — page 9
  should prescribe rest and recovery periods as mandatory
blocks of the overall training plan, irrespective of pressures
from sports coaches or parents. T o optimize physical devel-
opment and minimize accumulated fatigue,...
-----------------------------------------------------------------
Chunk #3 | SSW.pdf — page 2
  Who Wants to Be a Novice?
3 StartingStrength.com© 2013 The Aasgaard Company
so he’s not really capable of inflicting enough training stress in a sane workout to prevent his recovery 
in a short period of time. This time ...
---

In [27]:
# ── Query 4: Specific factual query ──────────────────────────────────────
answer4 = run_rag("How many sets and reps should a beginner do per exercise?")


  QUERY: How many sets and reps should a beginner do per exercise?

📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------

  Chunk #1 | NSCA_3.pdf — page 6 | 500 chars
  sets (2–3) per exercise.
Repetitions 8–12 or 10–15 Perform 6–12 reps with variation for muscular strength for healthy older
adults.
Perform 10–15 repetitions at a lower relative resistance for beginners.
Intensity 70–85%...

  Chunk #2 | SSW.pdf — page 2 | 431 chars
  heavy set. ONE. Really. Sets-across deadlifts do not work, because for the deadlift more is not better . 
T rust me on this.
Workout B is: Squat 3 sets of 5 across again, bench press 3 sets of 5 across, and power clean 5...

  Chunk #3 | SSW.pdf — page 2 | 426 chars
  curls, do them today, heavy, for 3 sets of 10. But I’d rather you wait at least a couple of months; your 
arms will grow a lot from the chins and deadlifts without doing a single curl, and they may interfere 
with recove...

  Chunk #4 | SSW.pdf — 

In [28]:
# ── Query 5: Ambiguous query (robustness test) ───────────────────────────
answer5 = run_rag("What is the best workout?")


  QUERY: What is the best workout?

📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------

  Chunk #1 | SSW.pdf — page 2 | 460 chars
  an empty bar doing sets of 5, and go up in small jumps. When you reach a weight that feels heavy, but 
not so heavy that your form has changed, stay there and do two more sets. The next workout, go up to 
a weight that i...

  Chunk #2 | SSW.pdf — page 2 | 485 chars
  frequency is not enough to make the best use of your potential to get big and strong as fast as possible. 
The idea is to train, rest 48 hours, train again, rest 48 hours, and train again, this time resting 72 hours 
so ...

  Chunk #3 | WHO.pdf — page 7 | 511 chars
  improvement or maintenance of one or more components of physical fitness is the objective. “Exercise” and “exercise 
training” frequently are used interchangeably and generally refer to physical activity performed during...

  Chunk #4 | NSCA_5.pdf — page 10 | 459 chars
  /C15Provid

In [33]:
# ── Query 6: Out-of-scope (hallucination boundary test) ──────────────────
answer6 = run_rag("What should I eat before a workout to maximise performance?")


  QUERY: What should I eat before a workout to maximise performance?

📂 RETRIEVED CONTEXT (5 chunks):
-----------------------------------------------------------------

  Chunk #1 | NSCA_5.pdf — page 10 | 504 chars
  proper exercise technique.
There are many ways to arrange the sequence of exercises
in a resistance training session. Most youth will perform total
body workouts several times per week, which involve
multiple exercises s...

  Chunk #2 | SSW.pdf — page 3 | 505 chars
  don’t eat and sleep enough. And again, by “most” I mean the vast, overwhelming majority. Like 95%. 
This period of growth cannot occur unless you create both the stress of heavy progressive barbell 
training and an envir...

  Chunk #3 | Increasing_Anaerobic_Endurance_Using_Strength_Endu.pdf — page 2 | 456 chars
  endurance training, hindering physical preparation for training focused on developing 
physiological qualities. Assessments that include components of muscle strength, such as speed 
or maximal oxyg

In [32]:
rag_config = {
    "llm_model"       : "claude-haiku-4-5-20251001",
    "temperature"     : 0,
    "max_tokens"      : 1024,
    "retriever"       : "mmr",
    "k"               : 5,
    "fetch_k"         : 20,
    "lambda_mult"     : 0.5,
    "embedding_model" : config["model_name"],
    "index_path"      : config["index_path"],
    "total_chunks"    : config["total_chunks"]
}

rag_config_path = "../data/processed/rag_config.json"
with open(rag_config_path, "w") as f:
    json.dump(rag_config, f, indent=2)

print("✅ RAG config saved to", rag_config_path)
print(json.dumps(rag_config, indent=2))


✅ RAG config saved to ../data/processed/rag_config.json
{
  "llm_model": "claude-haiku-4-5-20251001",
  "temperature": 0,
  "max_tokens": 1024,
  "retriever": "mmr",
  "k": 5,
  "fetch_k": 20,
  "lambda_mult": 0.5,
  "embedding_model": "multi-qa-MiniLM-L6-cos-v1",
  "index_path": "../embeddings/vector_store",
  "total_chunks": 2997
}
